## Задача 2 (дополнительная)

- Для кода с порождающей матрицей
  $$
  G = \left[\begin{array}{}
  1 & 0 & 1 & 1 & 0 & 1\\
  0 & 0 & 1 & 1 & 1 & 0\\
  0 & 1 & 0 & 1 & 1 & 1\\
  \end{array}
  \right]
  $$

построить три решётки по следующим алгоритмам

- по кодовым словам (т.е. не использовать свойство линейности кода)
- по порождающей матрице
- по проверочной матрице


In [1]:
from itertools import product


def vec_add(
    a, b
):  # побитовая сумма двух векторов по модулю 2 (сложение векторов в F_2)
    return tuple((x + y) % 2 for x, y in zip(a, b))


def dot(a, b):  # скалярное произведение по модулю 2 для векторов над F_2
    return (
        sum(x * y for x, y in zip(a, b)) % 2
    )  # возвращаем скалярное произведение по модулю 2


def rank_gf2(
    rows,
):  # подсчёт ранга матрицы над F_2 (по строкам), с приведением к ступенчатому виду методом Гаусса
    matrix = [list(row) for row in rows if any(row)]  # копируем только ненулевые строки
    if not matrix:
        return 0
    r = 0  # ранг матрицы
    n_rows = len(matrix)  # число строк
    n_cols = len(matrix[0])  # число столбцов
    for col in range(n_cols):
        pivot = None  # ведущий элемент
        for i in range(
            r, n_rows
        ):  # ищем строку с ненулевым элементом в текущем столбце, начиная с r-й
            if matrix[i][col] == 1:
                pivot = i  # запоминаем строку
                break
        if pivot is None:
            continue  # если в столбце нет 1, то переходим к следующему столбцу
        matrix[r], matrix[pivot] = (
            matrix[pivot],
            matrix[r],
        )  # меняем строки местами, чтобы ведущий элемент стоял на диагонали
        for i in range(
            n_rows
        ):  # зануляем все остальные элементы в текущем столбце (сложение по mod 2)
            if i != r and matrix[i][col] == 1:
                matrix[i] = [
                    (matrix[i][j] + matrix[r][j]) % 2 for j in range(n_cols)
                ]  # сложение по mod 2
        r += 1  # увеличиваем ранг
        if r == n_rows:  # если достигли последней строки, то выходим из цикла
            break
    return r  # возвращаем ранг матрицы


def format_vec(v):  # форматирование вектора в строку
    return "".join(map(str, v))


def all_codewords_from_G(G):  # перечисляем все кодовые слова по порождающей матрице
    k = len(G)  # число строк в порождающей матрице
    n = len(G[0])  # число столбцов в порождающей матрице
    words = []  # список всех кодовых слов
    for m in product(
        [0, 1], repeat=k
    ):  # перебираем все возможные комбинации коэффициентов
        c = []  # текущее кодовое слово
        for j in range(n):
            c.append(sum(m[i] * G[i][j] for i in range(k)) % 2)  # сложение по mod 2
        words.append(tuple(c))  # добавляем текущее кодовое слово в список
    return sorted(set(words))  # возвращаем список всех кодовых слов


def parity_check_from_G(G):  # строим проверочную матрицу по порождающей матрице
    n = len(G[0])
    orth = []  # список всех векторов, ортогональных порождающей матрице
    for v in product(
        [0, 1], repeat=n
    ):  # перебираем все возможные комбинации коэффициентов
        if all(
            dot(row, v) == 0 for row in G
        ):  # если вектор ортогонален порождающей матрице, то добавляем его в список
            orth.append(v)
    basis = []  # базис проверочной матрицы
    for v in orth:  # перебираем все векторы в базисе
        if not any(v):  # если вектор нулевой, то пропускаем
            continue
        if rank_gf2(basis + [v]) > rank_gf2(
            basis
        ):  # если ранг матрицы больше, то добавляем вектор в базис
            basis.append(v)
    return [list(v) for v in basis]  # возвращаем проверочную матрицу


def trellis_by_codewords(codewords):  # строим решётку по кодовым словам
    n = len(codewords[0])
    profile = []  # список числа узлов на каждом ярусе
    signatures = []  # список всех узлов на каждом ярусе
    for i in range(n + 1):  # перебираем все ярусы
        pref_to_future = {}  # словарь всех префиксов и будущих суффиксов
        for w in codewords:  # перебираем все кодовые слова
            pref_to_future.setdefault(w[:i], set()).add(
                w[i:]
            )  # добавляем префикс и суффикс в словарь
        uniq = {
            frozenset(suf_set) for suf_set in pref_to_future.values()
        }  # добавляем все уникальные суффиксы в словарь
        profile.append(len(uniq))  # добавляем число узлов на текущем ярусе
        signatures.append(
            sorted(uniq, key=lambda x: sorted(x))
        )  # добавляем все узлы на текущем ярусе
    return (
        profile,
        signatures,
    )  # возвращаем список числа узлов на каждом ярусе и список всех узлов на каждом ярусе


def row_span_interval(row):  # находим интервал, в котором строка активна
    ones = [i for i, bit in enumerate(row) if bit == 1]  # находим все единицы в строке
    return min(ones), max(ones) + 1  # возвращаем интервал, в котором строка активна


def profile_by_active_rows(G):  # строим профиль по активным строкам
    n = len(G[0])  # число столбцов в порождающей матрице
    spans = [row_span_interval(row) for row in G]
    # На ярусе i (от 0 до n) число узлов = 2^(число активных строк).
    # Строка активна на ярусе i, если отрезок от первой до последней единицы
    # в строке (см. минимальную спеновую форму) пересекает разрез между
    # уже прочитанным префиксом и остатком — эквивалентно s < i < e для [s, e).
    return [2 ** sum(1 for s, e in spans if s < i < e) for i in range(n + 1)]


def distinct_starts_ends(G):  # проверяем, что все начала и концы различны
    spans = [
        row_span_interval(row) for row in G
    ]  # находим интервалы, в которых строки активны
    starts = [s for s, _ in spans]  # находим все начала
    ends = [e for _, e in spans]  # находим все концы
    return len(set(starts)) == len(starts) and len(set(ends)) == len(
        ends
    )  # возвращаем, что все начала и концы различны


def mat_mul_gf2(A, B):  # умножаем две матрицы над F_2
    rows = len(A)  # число строк в первой матрице
    cols = len(B[0])  # число столбцов во второй матрице
    mid = len(B)  # число строк во второй матрице
    C = [[0] * cols for _ in range(rows)]  # создаём пустую матрицу
    for i in range(rows):  # перебираем все строки первой матрицы
        for j in range(cols):  # перебираем все столбцы второй матрицы
            C[i][j] = (
                sum(A[i][t] * B[t][j] for t in range(mid)) % 2
            )  # умножаем строку первой матрицы на столбец второй матрицы и добавляем результат в матрицу C
    return C  # возвращаем результат умножения


def invertible_matrices_3x3():  # перебираем все обратимые матрицы 3x3 над F_2
    mats = []
    for bits in product(
        [0, 1], repeat=9
    ):  # перебираем все возможные комбинации коэффициентов
        M = [
            list(bits[0:3]),
            list(bits[3:6]),
            list(bits[6:9]),
        ]  # создаём матрицу из 9 бит
        if (
            rank_gf2([tuple(row) for row in M]) == 3
        ):  # если ранг матрицы равен 3, то добавляем матрицу в список
            mats.append(M)
    return mats  # возвращаем список всех обратимых матриц 3x3 над F_2


def best_msf_equivalent(G, target_profile):
    best = None  # лучшая матрица
    for U in invertible_matrices_3x3():
        G2 = mat_mul_gf2(U, G)  # умножаем матрицу G на матрицу U
        if any(
            not any(row) for row in G2
        ):  # если в матрице G2 есть нулевая строка, то пропускаем
            continue
        if not distinct_starts_ends(
            G2
        ):  # если все начала и концы не различны, то пропускаем
            continue
        prof = profile_by_active_rows(G2)  # строим профиль по активным строкам
        score = (prof != target_profile, max(prof), prof)  # считаем оценку
        if (
            best is None or score < best[0]
        ):  # если лучшая матрица не найдена или оценка лучше, то обновляем лучшую матрицу
            best = (score, G2, prof)
    return best[1], best[2]  # возвращаем лучшую матрицу и профиль


def partial_syndrome(prefix, columns, r):  # считаем частичный синдром
    s = (0,) * r  # создаём нулевой вектор длины r
    for bit, col in zip(prefix, columns):  # перебираем все биты и столбцы
        if bit == 1:  # если бит равен 1, то добавляем столбец в вектор
            s = vec_add(s, col)  # добавляем столбец в вектор
    return s  # возвращаем вектор синдрома


def trellis_by_H(codewords, H):  # строим решётку по проверочной матрице
    n = len(codewords[0])  # длина кодового слова
    r = len(H)  # число строк в проверочной матрице
    cols = list(zip(*H))  # транспонируем проверочную матрицу
    profile = []  # список числа узлов на каждом ярусе
    signatures = []  # список всех узлов на каждом ярусе
    for i in range(n + 1):
        syn_to_future = {}  # словарь всех синдромов и будущих суффиксов
        for w in codewords:  # перебираем все кодовые слова
            syn = partial_syndrome(w[:i], cols[:i], r)
            syn_to_future.setdefault(syn, set()).add(
                w[i:]
            )  # добавляем синдром и суффикс в словарь
        profile.append(len(syn_to_future))  # добавляем число узлов на текущем ярусе
        uniq = {
            frozenset(suf_set) for suf_set in syn_to_future.values()
        }  # добавляем все уникальные суффиксы в словарь
        signatures.append(
            sorted(uniq, key=lambda x: sorted(x))
        )  # добавляем все узлы на текущем ярусе
    return (
        profile,
        signatures,
    )  # возвращаем список числа узлов на каждом ярусе и список всех узлов на каждом ярусе

In [2]:
G = [
    [1, 0, 1, 1, 0, 1],
    [0, 0, 1, 1, 1, 0],
    [0, 1, 0, 1, 1, 1],
]

codewords = all_codewords_from_G(
    G
)  # перечисляем все кодовые слова по порождающей матрице
H = parity_check_from_G(G)  # строим проверочную матрицу по порождающей матрице
profile_cw, sign_cw = trellis_by_codewords(
    codewords
)  # строим решётку по кодовым словам
G_msf, profile_g = best_msf_equivalent(
    G, profile_cw
)  # строим минимальную спеновую форму по порождающей матрице
profile_h, sign_h = trellis_by_H(codewords, H)  # строим решётку по проверочной матрице

print("Порождающая матрица G (как была дана в задаче):")
for row in G:  # перебираем все строки порождающей матрицы
    print("  ", row)

print("\nПорождающая матрица G в минимальной спеновой форме (МСФ):")
for row in G_msf:  # перебираем все строки минимальной спеновой формы
    print("  ", row)

print("\nПроверочная матрица H для этого кода:")
for row in H:  # перебираем все строки проверочной матрицы
    print("  ", row)

print("\nПрофиль решётки (число узлов на каждом ярусе) при разных построениях:")
print(f" - По кодовым словам:                    {profile_cw}")
print(f" - По порождающей матрице G в МСФ:        {profile_g}")
print(f" - Синдромная решётка по проверочной H:  {profile_h}")

if all(a == b for a, b in zip(sign_cw, sign_h)):
    print(
        "\nРешётки, построенные по кодовым словам и синдромная по H, совпадают "
        "с точностью до перенумерации узлов (изоморфны)."
    )
else:
    print(
        "\nРешётки по кодовым словам и синдромная по H различаются "
        "(не совпадают с точностью до перенумерации узлов)."
    )

if profile_cw == profile_g == profile_h:
    print("Профили всех трёх решёток совпадают между собой.")
else:
    print("Профили решёток получились разными.")

print("\nВсе возможные кодовые слова этого кода (по порядку):")
for i, w in enumerate(codewords):
    print(f"{i + 1:2d}: {format_vec(w)}")

Порождающая матрица G (как была дана в задаче):
   [1, 0, 1, 1, 0, 1]
   [0, 0, 1, 1, 1, 0]
   [0, 1, 0, 1, 1, 1]

Порождающая матрица G в минимальной спеновой форме (МСФ):
   [0, 1, 0, 1, 1, 1]
   [0, 0, 1, 1, 1, 0]
   [1, 1, 0, 1, 0, 0]

Проверочная матрица H для этого кода:
   [0, 0, 1, 0, 1, 1]
   [0, 1, 0, 1, 1, 1]
   [1, 0, 0, 1, 1, 0]

Профиль решётки (число узлов на каждом ярусе) при разных построениях:
 - По кодовым словам:                    [1, 2, 4, 8, 4, 2, 1]
 - По порождающей матрице G в МСФ:        [1, 2, 4, 8, 4, 2, 1]
 - Синдромная решётка по проверочной H:  [1, 2, 4, 8, 4, 2, 1]

Решётки, построенные по кодовым словам и синдромная по H, совпадают с точностью до перенумерации узлов (изоморфны).
Профили всех трёх решёток совпадают между собой.

Все возможные кодовые слова этого кода (по порядку):
 1: 000000
 2: 001110
 3: 010111
 4: 011001
 5: 100011
 6: 101101
 7: 110100
 8: 111010
